<a href="https://colab.research.google.com/github/anyuanay/INFO212/blob/main/INFO212_Week8_Lecture2_Aggretation_PivotTable_CrossTab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

## Week 8: Lecture 2: Data Aggregation and Group Operations
---

**Agenda:**
- Group by index level
- Group by user-defined functions
- pivotal tables and cross-tabulation


In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Grouping by Index Levels
## A convenience for hierarchically indexed datasets is the ability to aggregate using one of the levels of an axis index. Let’s look at an example: `Cleaned_Laptop_data.csv`

In [ ]:
from google.colab import files
files.upload()

## Explore the data

## Set up hierarchical index by brand, model, ram_gb

## To group by level, pass the level number or name using the level keyword:

## Aggregate on brand for mean latest_price

# Data Aggregation using User-Defined Functions
- You can use aggregations of your own function that is also defined on the grouped object.
- To use your own aggregation functions, pass any function that aggregates an **array/series** to the `aggregate` or `agg` method.

```
def peak_to_peak(ser):
    return ser.max() - ser.min()
```

# DataSet: Meals and Tips
Download the `tips.csv` dataset and load it as a DataFrame here.

In [ ]:
from google.colab import files
files.upload()

## Explore the tips dataset

## Add a tip_pct column
Compute the percentage of tips for each meal

## For each unique value in the day column, compute the maximum, mininum, and the difference between the max and min for total_bill and tip_pct.

## Select Top Results on Aggregation
Suppose we wanted to select the top five tip_pct values by group. First, write a function that selects the rows with the largest values in a particular column from a **DataFrame**:

```
def top(df, n=5, column='tip_pct'):
    return df.sort_values(by=column, ascending=False)[:n]

```

Now, if we group by smoker, say, and call apply with this function, we get the following:

```
tips.groupby('smoker').apply(top)
```

The top function is called on each row group from the
DataFrame, and then the results are glued together using pandas.concat, labeling the
pieces with the group names. The result therefore has a hierarchical index whose
inner level contains index values from the original DataFrame.

## For each unique value in the day column, extract the top 3 total bills.

# What is a Pivot Table and How to Use It?
- A pivot table is a data summarization tool frequently found in spreadsheet programs
and other data analysis software.
- It aggregates a table of data by one or more keys,
arranging the data in a rectangle with some of the group keys along the rows and
some along the columns.
- Pivot tables in Python with pandas are made possible
through the groupby facility combined with reshape operations
utilizing hierarchical indexing.
- DataFrame has a pivot_table method, and
there is also a top-level pandas.pivot_table function.
- In addition to providing a
convenience interface to groupby, pivot_table can add partial totals, also known as
margins.

## For the `tips` dataset, suppose you wanted to compute a table of group means (the default pivot_table aggregation type) arranged by day and smoker on the rows:
```
tips[['day', 'smoker', 'total_bill', 'tip']].pivot_table(index=['day', 'smoker'])
```

## Now, suppose we want to aggregate only tip_pct and size, and additionally group by time. I’ll put smoker in the table columns and day in the rows:
```
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker')
```

## We could augment this table to include partial totals by passing margins=True. This has the effect of adding All row and column labels, with corresponding values being the group statistics for all the data within a single tier:
```
tips.pivot_table(['size', 'tip_pct'], index=['time', 'day'],
                 columns='smoker', margins=True)
```

## Here, the All values are means without taking into account smoker versus nonsmoker (the All columns) or any of the two levels of grouping on the rows (the All row).

## To use a different aggregation function, pass it to aggfunc. For example, 'count' or len will give you a cross-tabulation (count or frequency) of group sizes:
```
tips.pivot_table('tip_pct', index=['time', 'smoker'], columns='day',
                 aggfunc=len, margins=True)
```

## If some combinations are empty (or otherwise NA), you may wish to pass a fill_value:
```
tips.pivot_table('tip_pct', index=['time', 'size', 'smoker'],
                 columns='day', aggfunc='mean', fill_value=0)
```

## pivot_table() is basically a convenient combination of groupby + aggregation + reshape.

# What is Cross-Tabulation? How to Use Crosstab?
## A cross-tabulation (or crosstab for short) is a special case of a pivot table that computes group frequencies. Here is an example:

In [ ]:
tips.columns

```
pd.crosstab(tips.day, tips.smoker, margins=True)
```

## Of course, it can be used to compute means for one series

```
pd.crosstab(tips.day, tips.smoker, tips['tip'], aggfunc='mean', margins=True)
```

## For more than one aggregation values, use pivot_table

```
pd.crosstab(tips.day, tips.smoker, tips['tip'], tips['total_bill'], aggfunc='mean', margins=True)
```

```
tips.pivot_table(['tip', 'total_bill'], index=['day'], columns=['smoker'], margins=True)
```